# Daily Journal 1

## Notebook Structure

1. [Introduction](#1-instroduction)
2. [Setup (Code)](#2-setup-code)
3. [Load Sample Data](#3-load-sample-data)
4. [Inspect Join Keys](#4-inspect-join-keys)
5. [Attempt 1: FIPS Join (Expected Failure)](#5-attempt-1-fips-join-expected-failure)
6. [Attempt 2: CBSA Join](#6-attempt-2-cbsa-join)
7. Analysis of Mismatches
8. Conclusions

## 1. Introduction

### Objective

The goal of this analysis is to integrate regional cost-of-living data from the BEA with occupational wage data from the BLS.

To achieve this, we attempt to join:
- BEA datasets (bea_mairpd, bea_marpp)
- BLS dataset (oews_msa)
- Census CBSA delineation file

We explore multiple join strategies and document the challenges encountered.

## 2. Setup (Code)

In [2]:
# 2. Setup

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="data-intensive-systems-final")

## 3. Load Sample Data

Don't load everything--just samples.

In [3]:
# 3. Load Sample Data

bea_query = """
    SELECT
        GeoFIPS,
        GeoName
    FROM `data-intensive-systems-final.affordability_pipeline.bea_mairpd`
    LIMIT 1000
"""

bls_query = """
    SELECT
        AREA,
        area_title
    FROM `data-intensive-systems-final.affordability_pipeline.oews_msa`
    LIMIT 1000
"""

census_query = """
    SELECT
        `CBSA Code`,
        `CBSA Title`,
        `FIPS State Code`,
        `Fips County Code`
    FROM `data-intensive-systems-final.affordability_pipeline.census_delineation_file`
    LIMIT 1000
"""

In [4]:
bea_df = client.query(bea_query).to_dataframe()
bls_df = client.query(bls_query).to_dataframe()
census_df = client.query(census_query).to_dataframe()

/home/mmercer/miniconda3/envs/bq/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 4. Inspect Join Keys

In [5]:
print("BEA sample:")
display(bea_df.head())

print("BLS sample:")
display(bls_df.head())

print("Census sample:")
display(census_df.head())

BEA sample:


,GeoFIPS,GeoName
0,00000,United States
1,00999,United States (Nonmetropolitan Portion) *
2,10180,"Abilene, TX (Metropolitan Statistical Area)"
3,10420,"Akron, OH (Metropolitan Statistical Area)"
4,10500,"Albany, GA (Metropolitan Statistical Area)"


BLS sample:


,AREA,area_title
0,10180,"Abilene, TX"
1,10180,"Abilene, TX"
2,10180,"Abilene, TX"
3,10180,"Abilene, TX"
4,10180,"Abilene, TX"


Census sample:


,CBSA Code,CBSA Title,FIPS State Code,Fips County Code
0,None,None,<NA>,<NA>
1,Note: The 2010 OMB Standards for Delineating M...,None,<NA>,<NA>
2,"Source: File prepared by U.S. Census Bureau, P...",None,<NA>,<NA>
3,Internet Release Date: April 2020,None,<NA>,<NA>
4,10700,"Albertville, AL",1,95


## 5. Attempt 1: FIPS Join (Expected Failure)

In [6]:
fips_join_query = """
    SELECT COUNT(*) AS match_count
    FROM `data-intensive-systems-final.affordability_pipeline.bea_mairpd` b
    JOIN `data-intensive-systems-final.affordability_pipeline.census_delineation_file` c
        ON TRIM(b.GeoFIPS) = (
            LPAD(CAST(c.`FIPS State Code` AS STRING), 2, '0') ||
            LPAD(CAST(C.`FIPS County Code` AS STRING), 3, '0')
        )
"""

fips_result = client.query(fips_join_query).to_dataframe()
display(fips_result)

/home/mmercer/miniconda3/envs/bq/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,match_count
0,0


### Observation

The join between BEa and Census using FIPS codes returns very few or zero matches.

### Possible Causes:
- BEA data ay not be strictly county-level
- Formatting inconsistencies in FIPS codes
- Presence of aggregated regions in BEA data

## 6. Attempt 2: CBSA Join

In [7]:
cbsa_join_query = """
    SELECT COUNT(*) AS match_count
    FROM `data-intensive-systems-final.affordability_pipeline.census_delineation_file` c
    JOIN `data-intensive-systems-final.affordability_pipeline.oews_msa` o
        ON CAST(c.`CNSA Code` AS STRING) = CAST(o.AREA AS STRING)
"""

cbsa_result = client.query(cbsa_join_query).to_dataframe()
display(cbsa_result)

BadRequest: 400 Name CNSA Code not found inside c at [5:19]; reason: invalidQuery, location: query, message: Name CNSA Code not found inside c at [5:19]

Location: US
Job ID: cce20703-1a6c-4028-888e-a93c6ca5ab16
